## 1. Setup

In [1]:
import sys
import os
import gc
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from dotenv import load_dotenv

project_root = r"c:\Users\admin\demand-forecasting"
load_dotenv(os.path.join(project_root, ".env"))
sys.path.append(project_root)


In [2]:
from src.data.loaders import load_parquet, load_many
from src.features.engineering import add_lag_features, add_rolling_mean
from src.models.train import prepare_features, train_lgbm
from src.models.evaluate import wape, mase, compute_bias


## 2. Chargement des données


In [3]:
full = load_parquet(project_root, "full")
print(full.shape)
print(full.columns.tolist())


(58327370, 13)
['item_id', 'store_id', 'quantite', 'la_date', 'lag_1', 'lag_7', 'rolling_mean_7', 'price', 'categorie', 'prix_connu', 'is_weekend', 'is_holiday', 'jour_semaine_num']


## 3. Split train / test

*Cutoff cohérent avec M4/M5/M6 : train < 2016-03-05, test = 50 derniers jours.*

In [4]:
cutoff_date = '2016-03-05'

train_fe = full[full['la_date'] < cutoff_date].copy()
test_fe = full[full['la_date'] >= cutoff_date].copy()

train_fe = train_fe.dropna(subset=['lag_1', 'lag_7'])

print(train_fe.shape)
print(test_fe.shape)


(56558950, 13)
(1554990, 13)


## 4. Préparation des features pour LightGBM

In [5]:
features = ['lag_1', 'lag_7', 'rolling_mean_7', 'price', 'prix_connu',
            'categorie', 'jour_semaine_num', 'is_weekend', 'is_holiday']
cat_features = ['categorie', 'jour_semaine_num', 'is_weekend', 'is_holiday']

X_train = prepare_features(train_fe, features, cat_features)
y_train = train_fe['quantite']

X_test = prepare_features(test_fe, features, cat_features)
y_test = test_fe['quantite']

print(X_train.dtypes)


lag_1                float64
lag_7                float64
rolling_mean_7       float64
price                float64
prix_connu             int64
categorie           category
jour_semaine_num    category
is_weekend          category
is_holiday          category
dtype: object


## 5. Entraînement du modèle final

*`alpha=0.63` (objectif quantile) — retenu en M6 pour corriger le biais de sous-prédiction sur les séries à fort volume (Q4), tout en restant sous le seuil WAPE < 0.70 fixé au cadrage M0.*

In [6]:
model = train_lgbm(X_train, y_train, cat_features, alpha=0.63, n_estimators=100, random_state=42)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.568759 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 706
[LightGBM] [Info] Number of data points in the train set: 56558950, number of used features: 9


## 6. Évaluation — WAPE global et biais

In [7]:
y_pred = model.predict(X_test).clip(min=0)

wape_score = wape(y_test, pd.Series(y_pred, index=y_test.index))
biais_global = compute_bias(y_test, pd.Series(y_pred, index=y_test.index))

print(f"WAPE : {wape_score:.4f}")
print(f"Biais global : {biais_global:.4f}")


WAPE : 0.6890
Biais global : -0.0397


## 7. Évaluation — WAPE par catégorie

In [8]:
test_fe['y_pred'] = y_pred

wape_par_categorie = test_fe.groupby('categorie').apply(
    lambda g: wape(g['quantite'], g['y_pred'])
)
print(wape_par_categorie)


categorie
FOODS        0.631485
HOBBIES      0.962907
HOUSEHOLD    0.746502
dtype: float64


## 8. Évaluation — MASE par série et par quartile de volume

*Le WAPE agrégé favorise les séries à fort volume. Le MASE, calculé par série, révèle la performance réelle sur la longue traîne.*

In [9]:
naive_scale = (
    train_fe.groupby(['item_id', 'store_id'])
    .apply(lambda g: (g['quantite'] - g['lag_1']).abs().mean())
    .rename('naive_scale')
)

mae_model = (
    test_fe.groupby(['item_id', 'store_id'])
    .apply(lambda g: (g['quantite'] - g['y_pred']).abs().mean())
    .rename('mae_model')
)

df_mase = pd.concat([mae_model, naive_scale], axis=1)
df_mase['mase'] = df_mase.apply(lambda r: mase(r['mae_model'], r['naive_scale']), axis=1)

df_mase['volume_quartile'] = pd.qcut(df_mase['naive_scale'], 4, labels=['Q1 (faible)', 'Q2', 'Q3', 'Q4 (fort)'])

print(df_mase.groupby('volume_quartile')['mase'].median())
print(df_mase.groupby('volume_quartile').apply(lambda g: (g['mase'] < 1).mean()))


volume_quartile
Q1 (faible)    1.475114
Q2             1.228075
Q3             1.031333
Q4 (fort)      0.864229
Name: mase, dtype: float64
volume_quartile
Q1 (faible)    0.366352
Q2             0.399002
Q3             0.472077
Q4 (fort)      0.650210
dtype: float64


## 9. Évaluation — biais par quartile de volume (segment critique : Q4)

In [10]:
biais_par_serie = test_fe.groupby(['item_id', 'store_id']).apply(
    lambda g: compute_bias(g['quantite'], g['y_pred'])
).rename('biais')

df_biais = df_mase[['volume_quartile']].join(biais_par_serie)
print(df_biais.groupby('volume_quartile')['biais'].mean())


volume_quartile
Q1 (faible)   -0.085991
Q2            -0.080719
Q3            -0.025831
Q4 (fort)      0.033897
Name: biais, dtype: float64


## 10. Backtesting multi-fenêtres

*Validation de la stabilité temporelle du modèle final sur 3 périodes de test distinctes (critère du cadrage M0 : dégradation < 15% entre fenêtres).*

In [11]:
test_windows = [
    ('2016-01-15', '2016-03-05'),
    ('2016-02-01', '2016-03-22'),
    ('2016-03-05', '2016-04-24'),
]

results_backtest = []

for train_end, test_end in test_windows:
    train_bt = full[full['la_date'] < train_end].dropna(subset=['lag_1', 'lag_7'])
    test_bt = full[(full['la_date'] >= train_end) & (full['la_date'] < test_end)]

    X_train_bt = prepare_features(train_bt, features, cat_features)
    y_train_bt = train_bt['quantite']
    X_test_bt = prepare_features(test_bt, features, cat_features)
    y_test_bt = test_bt['quantite']

    model_bt = train_lgbm(X_train_bt, y_train_bt, cat_features, alpha=0.63, n_estimators=100, random_state=42)

    y_pred_bt = model_bt.predict(X_test_bt).clip(min=0)
    wape_bt = wape(y_test_bt, pd.Series(y_pred_bt, index=y_test_bt.index))
    biais_bt = compute_bias(y_test_bt, pd.Series(y_pred_bt, index=y_test_bt.index))

    results_backtest.append({
        'train_end': train_end, 'test_end': test_end,
        'wape': wape_bt, 'biais_global': biais_bt,
        'n_train': len(train_bt), 'n_test': len(test_bt)
    })
    print(f"Train < {train_end} | Test [{train_end}, {test_end}) | WAPE={wape_bt:.4f} | biais={biais_bt:.4f}")

df_backtest = pd.DataFrame(results_backtest)
print(df_backtest)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 3.383613 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 707
[LightGBM] [Info] Number of data points in the train set: 55034450, number of used features: 9
Train < 2016-01-15 | Test [2016-01-15, 2016-03-05) | WAPE=0.6913 | biais=-0.0483
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.833239 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 708
[LightGBM] [Info] Number of data points in the train set: 55552780, number of used features: 9
Train < 2016-02-01 | Test [2016-02-01, 2016-03-22) | WAPE=0.6909 | biais=-0.0390
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 2.947466 seconds.
You can set `force_row_wis

## 11. Sauvegarde du modèle final

In [12]:
model_dir = os.path.join(project_root, "models")
os.makedirs(model_dir, exist_ok=True)
joblib.dump(model, os.path.join(model_dir, "lgbm_m6_quantile_alpha063.pkl"))


['c:\\Users\\admin\\demand-forecasting\\models\\lgbm_m6_quantile_alpha063.pkl']